# 12 — Retrospective April–June test evaluation

Run All in the matching saved environment. Score the published classifier on `experiments/holdout/05_modelling/test.parquet` with its existing cutoff, then compare predictions with observed next-week labels. The target is a local median rise of at least **10% in the next calendar week**.

April–June was previously inspected during development: this is a **retrospective test**, not an untouched independent evaluation.

Requires the active run, deployment pin, study/model, dashboard provenance and matching training/test artifacts. The longer-history baseline also needs its recorded 2024 source and cleaning files. Missing or changed inputs stop execution. Results are saved separately in `research/12_test_set_evaluation/`; no retraining, downloads or dashboard updates occur.

In [1]:
import ast
import hashlib
import json
import os
import sys
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score

REPOSITORY = Path.cwd().resolve()
if not (REPOSITORY / "pricecatcher_config.py").is_file():
    REPOSITORY = REPOSITORY.parent
if not (REPOSITORY / "pricecatcher_config.py").is_file():
    raise FileNotFoundError("Open the FYP project or research folder before running this notebook.")

In [2]:
RUN_DIRECTORY = None  # Normally leave unchanged; follows the existing active-run pointer.

## 1. Verify model and inputs

Check the existing pin, study/comparison hashes, published checkpoint, Python/package versions and test-file hash. Reuse only the model-input helper from main 12, without running its other cells or importing the settings module.

These checks establish artifact identity and split consistency. They do not undo prior data exposure or independently verify the original source prices.

In [3]:
def file_hash(path):
    with Path(path).open('rb') as handle:
        return hashlib.file_digest(handle, 'sha256').hexdigest()


def read_json(path):
    return json.loads(Path(path).read_text())


def inside(root, relative):
    path = (Path(root) / relative).resolve()
    if not path.is_relative_to(Path(root).resolve()):
        raise ValueError('A saved path escapes its expected folder.')
    return path


def calculations(run):
    """Reuse named read-only helpers without importing settings or executing notebooks."""
    namespace = {'Path': Path, 'hashlib': hashlib, 'json': json, 'np': np, 'pd': pd,
                 '__file__': str(REPOSITORY / 'pricecatcher_config.py'),
                 'PROJECT_ROOT': run, 'DATA_HOME': REPOSITORY / 'data'}
    tree = ast.parse((REPOSITORY / 'pricecatcher_config.py').read_text())
    mapping = next(node for node in tree.body if isinstance(node, ast.Assign)
                   and any(isinstance(t, ast.Name) and t.id == 'RESEARCH_NOTEBOOK_NAMES' for t in node.targets))
    names = {'installed_package_version', 'resolve_input', 'notebook_path', 'input_reference'}
    nodes = [mapping] + [node for node in tree.body if isinstance(node, ast.FunctionDef) and node.name in names]
    exec(compile(ast.Module(body=nodes, type_ignores=[]), 'read-only helpers', 'exec'), namespace)
    notebook = read_json(REPOSITORY / '12_shap_and_dashboard_inputs.ipynb')
    functions = [node for cell in notebook['cells'] if cell['cell_type'] == 'code'
                 for node in ast.parse(''.join(cell['source'])).body
                 if isinstance(node, ast.FunctionDef) and node.name == 'model_inputs']
    if len(functions) != 1:
        raise ValueError('Expected one saved model-input calculation in main Notebook 12.')
    exec(compile(ast.Module(body=functions, type_ignores=[]), 'model input preparation only', 'exec'), namespace)
    return namespace


def verify_inputs(run):
    """Require the existing published pin and its exact test-set identity; never create a pin."""
    protected = {}
    def remember(path, expected=None):
        path = Path(path).resolve()
        actual = file_hash(path)
        if expected is not None and actual != expected:
            raise ValueError(f'Frozen file changed: {path}. Restore the matching artifact; do not rewrite hashes.')
        protected[str(path)] = actual
        return path
    pin_path = run / '11_deployment/deployment_selection.json'
    if not pin_path.is_file():
        raise FileNotFoundError('Restore the existing deployment pin and matching run. No model or deployment is created here.')
    pin = read_json(remember(pin_path))
    study_path = remember(inside(run, pin['study_manifest']), pin['study_manifest_sha256'])
    comparison_path = remember(inside(run, pin['comparison_manifest']), pin['comparison_manifest_sha256'])
    study, comparison = read_json(study_path), read_json(comparison_path)
    declared = comparison['studies'][pin['candidate']]
    identity = {'study_sha256': pin['study_manifest_sha256'], 'cutoff': pin['score_cutoff']}
    if (study.get('status') != 'completed' or comparison.get('status') != 'completed'
            or comparison['candidate_under_fold_rule'] != pin['candidate']
            or inside(run, declared['folder']) != study_path.parent
            or declared['manifest_sha256'] != pin['study_manifest_sha256']
            or hashlib.sha256(json.dumps(identity, sort_keys=True).encode()).hexdigest()[:16] != pin['deployment_id']):
        raise ValueError('Pin, study and comparison identify different model selections.')
    selection = read_json(remember(study_path.parent / 'selection.json'))
    if selection != study['selection'] or selection['score_cutoff'] != pin['score_cutoff']:
        raise ValueError('Frozen selection or cutoff changed.')
    cutoff = float(selection['score_cutoff'])
    if not np.isfinite(cutoff) or not 0 <= cutoff <= 1 or selection['surge_threshold_pct'] != 10:
        raise ValueError('Expected the frozen score cutoff and existing 10% target.')
    for name, expected in comparison['output_sha256'].items():
        remember(inside(comparison_path.parent, name), expected)
    for name, expected in study['output_sha256'].items():
        remember(inside(study_path.parent, name), expected)
    for name, expected in study['input_sha256'].items():
        remember(inside(run, name), expected)
    helpers = calculations(run)
    for name, expected in study.get('additional_input_sha256', {}).items():
        remember(helpers['resolve_input'](name), expected)
    if sys.version_info[:2] != tuple(map(int, study.get('python_version', '3.12').split('.')[:2])):
        raise ValueError('Use the Python major/minor version recorded by this study.')
    for package, expected in study['package_versions'].items():
        if helpers['installed_package_version'](package) != expected:
            raise ValueError(f'Restore the saved {package} version before predicting.')
    prepared = run / 'experiments/holdout/05_modelling'
    settings_path = prepared / 'experiment_settings.json'
    settings = read_json(remember(settings_path, study['input_sha256'][settings_path.relative_to(run).as_posix()]))
    if (settings['train_end'], settings['validation_end'], settings['test_end']) != ('2025-12-31', '2026-03-31', '2026-06-30'):
        raise ValueError('This notebook evaluates only the original configured April–June 2026 test period.')
    if settings['surge_threshold_pct'] != 10 or selection['training_end'] != settings['train_end']:
        raise ValueError('Target or final training boundary differs from the frozen settings.')
    features = settings['feature_columns']
    if (features != selection['feature_columns'] or len(features) != len(set(features))
            or sorted(features) != sorted(settings['numeric_features'] + settings['categorical_features'])
            or any(name.startswith('next_') or name in {'surge_label', 'split'} for name in features)):
        raise ValueError('Frozen feature definitions differ or contain outcomes.')
    dashboard = run / '08_dashboard'
    pointer = dashboard / 'active_deployment.json'
    if pointer.is_file():
        record = read_json(remember(pointer))
        dashboard = inside(dashboard, record['directory'])
        if not dashboard.is_relative_to((run / '08_dashboard/deployments').resolve()):
            raise ValueError('Invalid dashboard deployment path.')
    entry = read_json(remember(dashboard / 'dashboard_manifest.json'))
    published = entry['forecast_model']
    model = remember(inside(run, published['model_path']), published['sha256'])
    if (published.get('deployment_id') != pin['deployment_id'] or published['score_cutoff'] != cutoff
            or model.parent != study_path.parent or study['output_sha256'].get(model.name) != published['sha256']):
        raise ValueError('The published dashboard checkpoint differs from the deployment pin.')
    cache = inside(dashboard, entry['cache_dir'])
    manifest = read_json(remember(cache / 'explanation_manifest.json', entry['cache_manifest_sha256']))
    if manifest['forecast_model'] != published:
        raise ValueError('Dashboard entry and cache describe different models.')
    test_path = prepared / 'test.parquet'
    expected_test = manifest['upstream_sha256'].get('prepared/holdout/test.parquet')
    if expected_test is None:
        raise ValueError('The original dashboard export does not record the test-set hash. Restore matching provenance; nothing is regenerated.')
    remember(test_path, expected_test)
    for path in [dashboard / 'refresh_manifest.json', run.parent.parent / 'active_run.txt']:
        if path.is_file():
            remember(path)
    for path in sorted((run / '13_registered_future_evaluation').glob('*/registration.json')):
        remember(path)
    for path in [REPOSITORY / 'pricecatcher_config.py', REPOSITORY / '12_shap_and_dashboard_inputs.ipynb']:
        remember(path)
    return pin, selection, settings, model, test_path, protected, helpers

## 2. Keep the test population and cutoff fixed

Match Notebook 05: the **current week end** must follow 31 March, and the **next-week outcome end** must be on or before 30 June. Check unique series-weeks and observed binary labels. Missing outcomes are not negatives.

Predict with the saved pipeline and cutoff. Where the following week's price also exists in the test table, recompute the 10% label; stop on a mismatch and report unchecked rows.

Compare the same rows with:
- **Always no surge:** constant ranking; AP equals prevalence when both classes occur.
- **Current-change rule:** rank by this week's price change, with alerts at 10%.
- **Training food-group rate:** estimate group rates from matching training history (2025 or 2024–2025); unseen groups use the overall training rate. Longer-history support is checked against 09b's saved audit. This is ranking only, so cutoff metrics are blank.

Undefined metrics are blank; AP is not assessed for single-class groups.

In [4]:
def validate_test(frame, settings):
    keys = ['item_code', 'state', 'district', 'premise_type', 'week_start']
    required = list(dict.fromkeys(settings['feature_columns'] + keys +
        ['item', 'unit', 'project_group', 'next_week_end', 'surge_label']))
    if frame.empty or set(required) - set(frame.columns):
        raise ValueError('Saved test rows are empty or missing required columns.')
    frame = frame[required].copy()
    for name in ['week_start', 'next_week_end']:
        frame[name] = pd.to_datetime(frame[name], errors='raise')
    if (frame[keys + ['next_week_end', 'project_group']].isna().any().any()
            or frame.duplicated(keys).any() or not frame.week_start.dt.day_of_week.eq(0).all()
            or not frame.next_week_end.eq(frame.week_start + pd.Timedelta(days=13)).all()):
        raise ValueError('Invalid dates, missing identifiers or duplicate local-series weeks.')
    # Match main 05 exactly: check CURRENT week end, not its Monday, against validation end.
    if (not (frame.week_start + pd.Timedelta(days=6)).gt(pd.Timestamp(settings['validation_end'])).all()
            or not frame.next_week_end.le(pd.Timestamp(settings['test_end'])).all()):
        raise ValueError('Test outcomes cross the frozen split boundaries.')
    if frame.surge_label.isna().any() or not frame.surge_label.isin([0, 1]).all():
        raise ValueError('Test outcomes must be observed binary surge labels; missing outcomes are not negatives.')
    if not np.isfinite(frame[settings['numeric_features']].fillna(0).to_numpy(dtype=float)).all():
        raise ValueError('Non-finite test features found. Missing features must retain the saved training-fitted preprocessing.')
    frame['surge_label'] = frame.surge_label.astype(int)
    frame['outcome_week'] = frame.week_start + pd.Timedelta(days=7)
    return frame


def metrics(frame):
    y, predicted = frame.surge_label.to_numpy(dtype=int), frame.predicted_surge.to_numpy(dtype=bool)
    tp = int((predicted & (y == 1)).sum()); fp = int((predicted & (y == 0)).sum())
    fn = int((~predicted & (y == 1)).sum()); tn = int((~predicted & (y == 0)).sum())
    return {'rows': len(frame), 'observed_surges': int(y.sum()), 'alerts': int(predicted.sum()),
        'surge_prevalence': float(y.mean()),
        'average_precision': float(average_precision_score(y, frame.model_score)) if len(np.unique(y)) == 2 else np.nan,
        'precision': tp / (tp + fp) if tp + fp else np.nan,
        'recall': tp / (tp + fn) if tp + fn else np.nan,
        'f1': 2 * tp / (2 * tp + fp + fn) if y.sum() else np.nan,
        'tp': tp, 'fp': fp, 'fn': fn, 'tn': tn}


def check_unchanged(protected, resolve=None):
    for name, expected in protected.items():
        path = resolve(name) if resolve is not None else Path(name)
        if file_hash(path) != expected:
            raise ValueError(f'Protected input changed: {name}. No completed result can be published.')

def baseline_comparison(frame, settings, group_rates=None, fallback=None):
    """Compare fixed rules on exactly the model's saved test population."""
    change = frame.weekly_price_change_pct.to_numpy(dtype=float)
    if not np.isfinite(change).all():
        raise ValueError('Current-change baseline requires finite previous-week changes; no rows are dropped.')
    tolerance = settings['preparation'].get('label_tolerance_pct', 0.0)
    rules = [
        ('Frozen selected model', frame.model_score.to_numpy(), frame.predicted_surge.to_numpy()),
        ('Always no surge', np.zeros(len(frame)), np.zeros(len(frame), dtype=bool)),
        ('Current-change rule (>=10%)', change,
         (change >= 10) | np.isclose(change, 10, atol=tolerance, rtol=0)),
    ]
    rows = []
    for method, scores, alerts in rules:
        result = metrics(frame.assign(model_score=scores, predicted_surge=alerts))
        result['ap_gain_over_prevalence'] = result['average_precision'] - result['surge_prevalence']
        rows.append({'method': method, **result})
    if group_rates is not None:
        scores = frame.project_group.map(group_rates).fillna(fallback).to_numpy(dtype=float)
        ap = float(average_precision_score(frame.surge_label, scores)) if frame.surge_label.nunique() == 2 else np.nan
        # Ranking reference only: no arbitrary classification cutoff is assigned.
        row = {key: np.nan for key in rows[0]}
        row.update(method='Training food-group rate (ranking only)', rows=len(frame),
            observed_surges=int(frame.surge_label.sum()), surge_prevalence=float(frame.surge_label.mean()),
            average_precision=ap, ap_gain_over_prevalence=ap - float(frame.surge_label.mean()),
            unseen_group_rows=int((~frame.project_group.isin(group_rates.index)).sum()))
        rows.append(row)
    return pd.DataFrame(rows)


def audit_test_targets(frame, settings):
    """Recompute only labels with an observed next calendar week in this saved table."""
    keys = ['item_code', 'state', 'district', 'premise_type']
    if not frame.weekly_median_price.gt(0).all():
        raise ValueError('Observed current prices must be positive.')
    following = frame[keys + ['week_start', 'weekly_median_price']].copy()
    following['week_start'] -= pd.Timedelta(days=7)
    following = following.rename(columns={'weekly_median_price': 'following_observed_price'})
    paired = frame.merge(following, on=keys + ['week_start'], how='left', validate='one_to_one')
    observed = paired.following_observed_price.notna()
    change = (paired.loc[observed, 'following_observed_price'] /
              paired.loc[observed, 'weekly_median_price'] - 1) * 100
    tolerance = settings['preparation'].get('label_tolerance_pct', 0.0)
    expected = (change >= 10) | np.isclose(change, 10, atol=tolerance, rtol=0)
    mismatches = int((expected.astype(int) != paired.loc[observed, 'surge_label']).sum())
    if mismatches:
        raise ValueError(f'{mismatches} saved labels disagree with consecutive observed test prices. Investigate preparation before reporting results.')
    return {'test_rows': len(frame), 'labels_recomputed': int(observed.sum()),
            'labels_without_next_price_in_test': int((~observed).sum()), 'label_mismatches': mismatches,
            'scope': 'saved consecutive test prices only; raw-source and coverage checks not repeated'}


def group_baseline_plan(run, pin, selection, protected, helpers):
    """Record exact training-only inputs before choosing the evaluation fingerprint."""
    study_path = inside(run, pin['study_manifest'])
    study = read_json(study_path)
    prepared = run / 'experiments/holdout/05_modelling'
    training_start = selection.get('training_start', '2025-01')
    if training_start not in {'2025-01', '2024-01'}:
        raise ValueError('Unsupported training history; do not silently substitute a baseline.')
    plan = {'history': training_start, 'train': str(prepared / 'train.parquet')}
    def require(path, expected):
        path = Path(path).resolve()
        if expected is None or file_hash(path) != expected:
            raise ValueError(f'Missing or changed frozen baseline input: {path}')
        protected[str(path)] = expected
        return str(path)
    require(plan['train'], study['input_sha256'].get('experiments/holdout/05_modelling/train.parquet'))
    if training_start == '2024-01':
        recorded = {str(helpers['resolve_input'](name).resolve()): digest
                    for name, digest in study['additional_input_sha256'].items()}
        sources = [name for name in recorded if Path(name).name == 'selected_prices_2024.parquet']
        if len(sources) != 1:
            raise ValueError('Restore the single recorded 2024 source cache. No downloading is performed.')
        stage3 = prepared.parent / '03_cleaned'
        for key, path in [('earlier', Path(sources[0])), ('daily', stage3 / 'daily_prices.parquet'),
                          ('cleaning', stage3 / 'cleaning_summary.json')]:
            plan[key] = require(path, recorded.get(str(path.resolve())))
        plan['audit'] = require(study_path.parent / 'training_comparison.csv',
                                study['output_sha256'].get('training_comparison.csv'))
        calculation = REPOSITORY / '09b_longer_history_lightgbm.ipynb'
        plan['calculation'] = require(calculation, file_hash(calculation))
    return plan


def training_group_rates(plan, settings):
    """Rebuild only frozen training labels when history was added; never fit a classifier."""
    from types import SimpleNamespace
    training = pd.read_parquet(plan['train'])
    if (training.empty or training.next_week_end.isna().any()
            or not pd.to_datetime(training.next_week_end).le(pd.Timestamp(settings['train_end'])).all()):
        raise ValueError('Baseline training outcomes cross the training boundary.')
    if plan['history'] == '2024-01':
        keys = ['project_group', 'item_code', 'item', 'unit', 'state', 'district', 'premise_type']
        columns = keys + ['date', 'premise_code', 'price']
        earlier = pd.read_parquet(plan['earlier'], columns=columns)
        earlier['date'] = pd.to_datetime(earlier.date)
        if earlier.empty or not earlier.date.between('2024-01-01', '2024-12-31').all():
            raise ValueError('Frozen historical source must contain 2024 records only.')
        cleaning = read_json(plan['cleaning'])
        if cleaning['preparation'] != settings['preparation']:
            raise ValueError('Cleaning and modelling preparation differ.')
        keep = earlier.price.notna() & np.isfinite(earlier.price) & earlier.price.gt(0)
        keep &= earlier[keys].notna().all(axis=1)
        if settings['preparation']['remove_borong']:
            keep &= ~earlier.premise_type.eq('Borong').fillna(False)
        keep &= ~earlier.item_code.isin(cleaning['excluded_item_codes'])
        end = pd.Timestamp(settings['train_end'])
        daily = pd.read_parquet(plan['daily'], columns=columns, filters=[('date', '<=', end)])
        combined = pd.concat([earlier.loc[keep], daily], ignore_index=True)
        combined['date'] = pd.to_datetime(combined.date)
        if combined.date.isna().any() or combined.date.gt(end).any() or combined.duplicated(['date', 'premise_code', 'item_code']).any():
            raise ValueError('Historical baseline source duplicates or crosses the training boundary.')
        functions = [node for cell in read_json(plan['calculation'])['cells'] if cell['cell_type'] == 'code'
                     for node in ast.parse(''.join(cell['source'])).body
                     if isinstance(node, ast.FunctionDef) and node.name == 'training_table']
        if len(functions) != 1:
            raise ValueError('Expected exactly one 09b training-table calculation.')
        tolerance = settings['preparation'].get('label_tolerance_pct', 0.0)
        cfg = SimpleNamespace(GROUP_KEYS=keys, MIN_RECORDS=settings['minimum_records'],
            MIN_PREMISES=settings['minimum_premises'], MIN_HISTORY_4W=settings['minimum_history_4w'],
            surge_mask=lambda change, threshold, tolerance_pct=tolerance:
                (change >= threshold) | np.isclose(change, threshold, atol=tolerance_pct, rtol=0))
        namespace = {'pd': pd, 'np': np, 'cfg': cfg, 'keys': keys + ['week_start'],
                     'training_end': end, 'settings': settings}
        exec(compile(ast.Module(body=functions, type_ignores=[]), '09b training labels only', 'exec'), namespace)
        longer = namespace['training_table'](combined, '2024-01')
        audit = pd.read_csv(plan['audit'])
        expected = audit.loc[audit.fold.eq('holdout') & audit.history.eq('2024-added')]
        if (len(expected) != 1 or len(longer) != int(expected.iloc[0].rows)
                or int(longer.surge_label.sum()) != int(expected.iloc[0].positives)):
            raise ValueError('Reconstructed baseline differs from the saved longer-history training audit.')
        common_keys = ['item_code', 'state', 'district', 'premise_type', 'week_start']
        common = training[common_keys + ['surge_label']].merge(longer[common_keys + ['surge_label']],
            on=common_keys, how='left', validate='one_to_one', suffixes=('', '_longer'))
        if not common.surge_label.eq(common.surge_label_longer).all():
            raise ValueError('Reconstructed history lost or changed original training labels.')
        training = longer
    if (training.project_group.isna().any() or training.surge_label.isna().any()
            or not training.surge_label.isin([0, 1]).all()):
        raise ValueError('Baseline needs observed binary training labels and food groups.')
    rates = training.groupby('project_group').surge_label.agg(rows='size', surges='sum', training_rate='mean').reset_index()
    return rates, float(training.surge_label.mean())


## 3. Save results

The fingerprint includes model, test file, cutoff and evaluation code. Completed matching results are verified and reused. Partial folders stop execution for inspection; earlier evaluations remain intact.

Save row-level predictions, overall/weekly/group metrics, baseline comparisons, training group rates, confusion counts, label audit and a manifest. Recheck protected inputs before publishing.

In [5]:
def evaluate_test(run):
    pin, selection, settings, model, test_path, protected, helpers = verify_inputs(run)
    plan = group_baseline_plan(run, pin, selection, protected, helpers)
    notebook = read_json(REPOSITORY / 'research/12_test_set_evaluation.ipynb')
    code = '\n'.join(''.join(cell['source']) for cell in notebook['cells']
                     if cell['cell_type'] == 'code' and 'configuration' not in cell.get('metadata', {}).get('tags', []))
    identity = {'deployment_id': pin['deployment_id'], 'candidate': pin['candidate'],
        'model_sha256': file_hash(model), 'test_sha256': file_hash(test_path),
        'verification_sha256': hashlib.sha256(json.dumps(protected, sort_keys=True).encode()).hexdigest(),
        'score_cutoff': float(selection['score_cutoff']), 'calculation_sha256': hashlib.sha256(code.encode()).hexdigest(),
        'evaluation_status': 'retrospective test-set evaluation; April–June was previously exposed during development'}
    evaluation_id = hashlib.sha256(json.dumps(identity, sort_keys=True).encode()).hexdigest()[:16]
    output = inside(run, 'research/12_test_set_evaluation') / evaluation_id
    saved_manifest = output / 'evaluation_manifest.json'
    if saved_manifest.is_file():
        saved = read_json(saved_manifest)
        if saved['identity'] != identity:
            raise ValueError('Existing evaluation belongs to different inputs.')
        check_unchanged(saved['input_sha256'], helpers['resolve_input'])
        for name, expected in saved['output_sha256'].items():
            if file_hash(inside(output, name)) != expected:
                raise ValueError('Saved evaluation output changed. Restore it; nothing is overwritten.')
        print('Verified existing test evaluation; reusing results, not another independent test.')
        return output
    if output.exists():
        raise ValueError('Partial evaluation folder exists. Preserve and inspect it; no automatic deletion or overwrite.')
    frame = validate_test(pd.read_parquet(test_path), settings)
    target_audit = audit_test_targets(frame, settings)
    rates, fallback = training_group_rates(plan, settings)
    group_rates = rates.set_index('project_group').training_rate
    pipeline = joblib.load(model)
    if list(pipeline.classes_) != [0, 1]:
        raise ValueError('Expected the saved binary 0/1 classifier.')
    scores = pipeline.predict_proba(helpers['model_inputs'](frame, settings))[:, 1]
    if len(scores) != len(frame) or not np.isfinite(scores).all() or not ((scores >= 0) & (scores <= 1)).all():
        raise ValueError('The frozen model returned invalid test scores.')
    frame['model_score'] = scores
    frame['predicted_surge'] = scores >= identity['score_cutoff']
    frame['outcome'] = np.select([frame.predicted_surge & frame.surge_label.eq(1),
        frame.predicted_surge & frame.surge_label.eq(0), ~frame.predicted_surge & frame.surge_label.eq(1)],
        ['correct_alert', 'false_alert', 'missed_surge'], default='correct_no_alert')
    overall = pd.DataFrame([metrics(frame)])
    baselines = baseline_comparison(frame, settings, group_rates, fallback)
    groups = pd.DataFrame([{'project_group': group, **metrics(part)} for group, part in frame.groupby('project_group', sort=True)])
    group_baselines = pd.concat([baseline_comparison(part, settings, group_rates, fallback).assign(project_group=group)
        for group, part in frame.groupby('project_group', sort=True)], ignore_index=True)
    weekly = pd.DataFrame([{'outcome_week': str(week.date()), **metrics(part)} for week, part in frame.groupby('outcome_week', sort=True)])
    m = overall.iloc[0]
    confusion = pd.DataFrame([[int(m.tn), int(m.fp)], [int(m.fn), int(m.tp)]],
        index=['Actual no surge', 'Actual surge'], columns=['Predicted no surge', 'Predicted surge'])
    check_unchanged(protected)
    output.mkdir(parents=True, exist_ok=False)
    columns = ['item_code', 'item', 'unit', 'project_group', 'state', 'district', 'premise_type',
               'week_start', 'outcome_week', 'next_week_end', 'surge_label', 'model_score', 'predicted_surge', 'outcome']
    frame[columns].to_parquet(output / 'test_predictions.parquet', index=False)
    frame[columns].to_csv(output / 'test_predictions.csv', index=False)
    for filename, table in [('overall_metrics.csv', overall), ('food_group_metrics.csv', groups), ('weekly_metrics.csv', weekly), ('baseline_metrics.csv', baselines), ('food_group_baseline_metrics.csv', group_baselines)]:
        table.to_csv(output / filename, index=False)
    rates.to_csv(output / 'training_food_group_rates.csv', index=False)
    confusion.to_csv(output / 'confusion_matrix.csv', index_label='Actual outcome')
    (output / 'target_audit.json').write_text(json.dumps(target_audit, indent=2))
    check_unchanged(protected)
    references = {}
    for name, expected in protected.items():
        try:
            reference = helpers['input_reference'](name)
        except ValueError:
            reference = name  # Preserve an explicitly recorded external legacy input.
        references[reference] = expected
    saved = {'status': 'completed', 'identity': identity, 'python_version': sys.version.split()[0],
        'completed_at': pd.Timestamp.now(tz='Asia/Kuala_Lumpur').isoformat(),
        'configured_period': {'after': settings['validation_end'], 'through': settings['test_end']},
        'actual_outcome_weeks': {'first': str(frame.outcome_week.min().date()), 'last': str(frame.outcome_week.max().date()),
                                 'weeks': int(frame.outcome_week.nunique())},
        'test_rows': len(frame), 'input_sha256': references,
        'output_sha256': {path.name: file_hash(path) for path in sorted(output.iterdir()) if path.is_file()},
        'target_audit': target_audit,
        'baseline_policy': {'training_history': plan['history'], 'training_rows': int(rates.rows.sum()),
            'fallback_rate': fallback, 'food_group_rate': 'ranking only; unseen groups use overall training rate',
            'scope': 'same test rows; no test tuning, refitting or cutoff selection'},
        'training_performed': False, 'threshold_selected_on_test': False, 'dashboard_modified': False,
        'limitations': ['previous exposure means this is not an untouched independent final test',
                        'metrics apply to supported saved test rows, not every shop or district',
                        'weeks and local series are dependent; no independent-row uncertainty claim',
                        'single-class AP and undefined precision/recall/F1 are blank',
                        'scores are uncalibrated, not verified probabilities']}
    with saved_manifest.open('x') as handle:
        json.dump(saved, handle, indent=2)
    return output

## 4. Run and view outputs

**Restart Kernel → Run All** with the saved local artifacts. The notebook prints its result folder and displays the tables. Data and fitted models are excluded from Git.

Workflow: **10 compare → 11 freeze deployment → main 12 export → research 12 evaluate**. Main 13 refreshes observations separately. The dashboard displays matching saved test results; it never runs this evaluation itself.

In [6]:
if RUN_DIRECTORY is not None:
    RUN = Path(RUN_DIRECTORY).expanduser().resolve()
elif os.environ.get("PRICECATCHER_DATA_DIR"):
    RUN = Path(os.environ["PRICECATCHER_DATA_DIR"]).expanduser().resolve()
else:
    pointer = REPOSITORY / "data" / "active_run.txt"
    if not pointer.is_file():
        raise FileNotFoundError("Restore data/active_run.txt and its matching saved run, or set RUN_DIRECTORY. No outputs were created.")
    relative = Path(pointer.read_text().strip())
    if relative.is_absolute() or relative.parts[:1] != ("runs",) or ".." in relative.parts:
        raise ValueError("Invalid active-run pointer.")
    RUN = inside(REPOSITORY / "data", relative)

OUTPUT = evaluate_test(RUN)
record = read_json(OUTPUT / "evaluation_manifest.json")
print("Model:", record["identity"]["candidate"])
print("Frozen cutoff:", record["identity"]["score_cutoff"])
print("Evidence status:", record["identity"]["evaluation_status"])
print("Test rows:", record["test_rows"], "| outcome weeks:", record["actual_outcome_weeks"])
print("Results saved to:", OUTPUT)
print("No retraining, cutoff selection, dashboard updates or registration changes.")
for filename in ["overall_metrics.csv", "baseline_metrics.csv", "training_food_group_rates.csv", "confusion_matrix.csv", "food_group_metrics.csv", "food_group_baseline_metrics.csv", "weekly_metrics.csv"]:
    print(filename)
    display(pd.read_csv(OUTPUT / filename))
print("First 10 predictions (all rows are saved in test_predictions.csv/parquet):")
display(pd.read_parquet(OUTPUT / "test_predictions.parquet").head(10))
print("Target check (limited to available consecutive test prices):")
display(pd.DataFrame([record["target_audit"]]))
comparison = pd.read_csv(OUTPUT / "baseline_metrics.csv").set_index("method")
model_ap = comparison.loc["Frozen selected model", "average_precision"]
prevalence = comparison.loc["Frozen selected model", "surge_prevalence"]
rule_ap = comparison.loc["Current-change rule (>=10%)", "average_precision"]
print(f"Test surge prevalence: {prevalence:.3f}; frozen model AP: {model_ap:.3f}; current-change AP: {rule_ap:.3f}")
print(f"AP difference versus prevalence: {model_ap - prevalence:+.3f}; versus current-change: {model_ap - rule_ap:+.3f}")
print("These are comparisons within this retrospective test period, not proof of improvement over validation.")

group_ap = comparison.loc["Training food-group rate (ranking only)", "average_precision"]
print(f"History-matched food-group baseline AP: {group_ap:.3f}; model AP difference: {model_ap - group_ap:+.3f}")
print("Food-group baseline ranks by training rates only; classification metrics are intentionally blank.")


Model: Longer-history LightGBM (tuned)
Frozen cutoff: 0.22099133987116373
Evidence status: retrospective test-set evaluation; April–June was previously exposed during development
Test rows: 57456 | outcome weeks: {'first': '2026-04-06', 'last': '2026-06-22', 'weeks': 12}
Results saved to: /Users/alfredyap/Documents/GitHub/FYP/data/runs/run_20261019_013241_618240/research/12_test_set_evaluation/79d295178b7dea41
No retraining, cutoff selection, dashboard updates or registration changes.
overall_metrics.csv


,rows,observed_surges,alerts,surge_prevalence,average_precision,precision,recall,f1,tp,fp,fn,tn
0,57456,7125,9968,0.124008,0.384115,0.373997,0.523228,0.436202,3728,6240,3397,44091


baseline_metrics.csv


,method,rows,observed_surges,alerts,surge_prevalence,average_precision,precision,recall,f1,tp,fp,fn,tn,ap_gain_over_prevalence,unseen_group_rows
0,Frozen selected model,57456,7125,9968.0,0.124008,0.384115,0.373997,0.523228,0.436202,3728.0,6240.0,3397.0,44091.0,0.260107,NaN
1,Always no surge,57456,7125,0.0,0.124008,0.124008,NaN,0.000000,0.000000,0.0,0.0,7125.0,50331.0,0.000000,NaN
2,Current-change rule (>=10%),57456,7125,7112.0,0.124008,0.150528,0.225394,0.224982,0.225188,1603.0,5509.0,5522.0,44822.0,0.026520,NaN
3,Training food-group rate (ranking only),57456,7125,NaN,0.124008,0.225126,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.101118,0.0


training_food_group_rates.csv


,project_group,rows,surges,training_rate
0,Chicken,40135,581,0.014476
1,Cooking Oil,25591,17,0.000664
2,Eggs,80379,1712,0.021299
3,Flour,16517,160,0.009687
4,Rice,9991,268,0.026824
5,Sugar,23398,0,0.000000
6,Vegetables,199746,36091,0.180684


... (9 outputs left out)

,item_code,item,unit,project_group,state,district,premise_type,week_start,outcome_week,next_week_end,surge_label,model_score,predicted_surge,outcome
0,1,AYAM BERSIH - STANDARD,1kg,Chicken,Johor,Batu Pahat,Hypermarket,2026-03-30,2026-04-06,2026-04-12,0,0.233275,True,false_alert
1,1,AYAM BERSIH - STANDARD,1kg,Chicken,Johor,Batu Pahat,Hypermarket,2026-04-06,2026-04-13,2026-04-19,0,0.116820,False,correct_no_alert
2,1,AYAM BERSIH - STANDARD,1kg,Chicken,Johor,Batu Pahat,Hypermarket,2026-04-13,2026-04-20,2026-04-26,0,0.048779,False,correct_no_alert
3,1,AYAM BERSIH - STANDARD,1kg,Chicken,Johor,Batu Pahat,Hypermarket,2026-04-20,2026-04-27,2026-05-03,0,0.071758,False,correct_no_alert
4,1,AYAM BERSIH - STANDARD,1kg,Chicken,Johor,Batu Pahat,Hypermarket,2026-04-27,2026-05-04,2026-05-10,0,0.018591,False,correct_no_alert
5,1,AYAM BERSIH - STANDARD,1kg,Chicken,Johor,Batu Pahat,Hypermarket,2026-05-04,2026-05-11,2026-05-17,0,0.046529,False,correct_no_alert
6,1,AYAM BERSIH - STANDARD,1kg,Chicken,Johor,Batu Pahat,Hypermarket,2026-05-11,2026-05-18,2026-05-24,0,0.032762,False,correct_no_alert
7,1,AYAM BERSIH - STANDARD,1kg,Chicken,Johor,Batu Pahat,Hypermarket,2026-05-18,2026-05-25,2026-05-31,0,0.005818,False,correct_no_alert
8,1,AYAM BERSIH - STANDARD,1kg,Chicken,Johor,Batu Pahat,Hypermarket,2026-05-25,2026-06-01,2026-06-07,0,0.006846,False,correct_no_alert
9,1,AYAM BERSIH - STANDARD,1kg,Chicken,Johor,Batu Pahat,Hypermarket,2026-06-01,2026-06-08,2026-06-14,0,0.006362,False,correct_no_alert


Target check (limited to available consecutive test prices):


,test_rows,labels_recomputed,labels_without_next_price_in_test,label_mismatches,scope
0,57456,50493,6963,0,saved consecutive test prices only; raw-source...


Test surge prevalence: 0.124; frozen model AP: 0.384; current-change AP: 0.151
AP difference versus prevalence: +0.260; versus current-change: +0.234
These are comparisons within this retrospective test period, not proof of improvement over validation.
History-matched food-group baseline AP: 0.225; model AP difference: +0.159
Food-group baseline ranks by training rates only; classification metrics are intentionally blank.


## Report notes

Report test AP beside prevalence and the baseline APs, followed by precision, recall, F1 and confusion counts at the frozen cutoff. Include every food group's positive support and the actual outcome-week range.

Disclose prior April–June exposure and the label audit's checked/unchecked counts. A higher AP than validation does not establish model improvement: the periods differ. The audit checks saved-price consistency, not all raw source records. Scores are uncalibrated; baseline change scores are percentages used for ranking.